# Silver Layer: Weather Conditions Transformation

This notebook transforms raw weather data from Bronze to Silver layer.

## Transformations Applied:
- **Column Standardization**: All columns converted to snake_case
- **Weather Condition Standardization**: Clean and categorize weather conditions (Clear, Cloudy, Rain, Storm, Snow, Fog, etc.)
- **Metric Validation**: Ensure values are within reasonable ranges:
  - Temperature: -50°C to 60°C
  - Humidity: 0-100%
  - Precipitation: >= 0 mm
  - Wind Speed: >= 0 km/h
- **Location Validation**: Ensure coordinates are within valid ranges and NSW bounds
- **Timestamp Validation**: Ensure timestamps are valid and not in the future
- **Deduplication**: Remove duplicate records based on station + timestamp

## Data Quality Validations:
- Count of records with invalid metrics
- Count of records with missing station info
- Weather condition distribution
- Temperature, humidity, and wind speed statistics
- Duplicate detection

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Table paths
BRONZE_TABLE = "mobility_ai.bronze.weather_raw"
SILVER_TABLE = "mobility_ai.silver.weather_conditions"

# NSW approximate bounds for location validation
NSW_LAT_MIN, NSW_LAT_MAX = -37.5, -28.0
NSW_LON_MIN, NSW_LON_MAX = 140.0, 154.0

print(f"Source: {BRONZE_TABLE}")
print(f"Target: {SILVER_TABLE}")

In [0]:
# Load bronze data
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze record count: {bronze_df.count()}")

# Standardize column names to snake_case (already in correct format, but keeping for consistency)
def to_snake_case(col_name):
    import re
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return re.sub(r'[\s-]+', '_', s2).lower()

# Rename all columns
for col in bronze_df.columns:
    bronze_df = bronze_df.withColumnRenamed(col, to_snake_case(col))

# Standardize weather condition (trim, title case)
silver_df = bronze_df.withColumn(
    "weather_condition_clean",
    F.initcap(F.trim(F.col("weather_condition")))
)

# Categorize weather conditions into standard groups
silver_df = silver_df.withColumn(
    "weather_category",
    F.when(F.col("weather_condition_clean").rlike("(?i)(clear|sunny|fine)"), "Clear")
     .when(F.col("weather_condition_clean").rlike("(?i)(cloud|overcast|partly)"), "Cloudy")
     .when(F.col("weather_condition_clean").rlike("(?i)(rain|shower|drizzle)"), "Rain")
     .when(F.col("weather_condition_clean").rlike("(?i)(storm|thunder|lightning)"), "Storm")
     .when(F.col("weather_condition_clean").rlike("(?i)(snow|sleet|ice|hail)"), "Snow/Ice")
     .when(F.col("weather_condition_clean").rlike("(?i)(fog|mist|haze)"), "Fog")
     .when(F.col("weather_condition_clean").rlike("(?i)(wind|gust|gale)"), "Windy")
     .otherwise("Other")
)

# Add data quality flags
silver_df = silver_df.withColumn(
    "has_valid_temperature",
    F.when(
        (F.col("temperature_celsius").isNotNull()) & 
        (F.col("temperature_celsius").between(-50, 60)),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_humidity",
    F.when(
        (F.col("humidity_percent").isNotNull()) & 
        (F.col("humidity_percent").between(0, 100)),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_precipitation",
    F.when(
        (F.col("precipitation_mm").isNotNull()) & 
        (F.col("precipitation_mm") >= 0),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_wind_speed",
    F.when(
        (F.col("wind_speed_kmh").isNotNull()) & 
        (F.col("wind_speed_kmh") >= 0),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_location",
    F.when(
        (F.col("location_lat").isNotNull()) & 
        (F.col("location_lon").isNotNull()) &
        (F.col("location_lat").between(NSW_LAT_MIN, NSW_LAT_MAX)) &
        (F.col("location_lon").between(NSW_LON_MIN, NSW_LON_MAX)),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_timestamp",
    F.when(
        (F.col("timestamp").isNotNull()) & 
        (F.col("timestamp") <= F.current_timestamp()),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_station",
    F.when(
        (F.col("station_id").isNotNull()) & 
        (F.length(F.col("station_id")) > 0),
        True
    ).otherwise(False)
)

# Deduplicate based on station_id + timestamp
window_spec = Window.partitionBy("station_id", "timestamp").orderBy(F.col("ingestion_timestamp").desc())

silver_df = silver_df.withColumn("row_num", F.row_number().over(window_spec))
silver_df = silver_df.filter(F.col("row_num") == 1).drop("row_num")

# Add processing metadata
silver_df = silver_df.withColumn("silver_processed_at", F.current_timestamp())

print(f"Silver record count after transformations: {silver_df.count()}")
display(silver_df.limit(10))

In [0]:
# Write to silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"✅ Successfully wrote {silver_df.count()} records to {SILVER_TABLE}")

In [0]:
# Validate silver data
validation_df = spark.table(SILVER_TABLE)

# Quality metrics
total_count = validation_df.count()
invalid_temp = validation_df.filter(F.col("has_valid_temperature") == False).count()
invalid_humidity = validation_df.filter(F.col("has_valid_humidity") == False).count()
invalid_precip = validation_df.filter(F.col("has_valid_precipitation") == False).count()
invalid_wind = validation_df.filter(F.col("has_valid_wind_speed") == False).count()
invalid_location = validation_df.filter(F.col("has_valid_location") == False).count()
invalid_timestamp = validation_df.filter(F.col("has_valid_timestamp") == False).count()
missing_station = validation_df.filter(F.col("has_valid_station") == False).count()

print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"Total Records: {total_count}")
print(f"Invalid Temperature: {invalid_temp} ({invalid_temp/total_count*100:.2f}%)")
print(f"Invalid Humidity: {invalid_humidity} ({invalid_humidity/total_count*100:.2f}%)")
print(f"Invalid Precipitation: {invalid_precip} ({invalid_precip/total_count*100:.2f}%)")
print(f"Invalid Wind Speed: {invalid_wind} ({invalid_wind/total_count*100:.2f}%)")
print(f"Invalid Location: {invalid_location} ({invalid_location/total_count*100:.2f}%)")
print(f"Invalid Timestamp: {invalid_timestamp} ({invalid_timestamp/total_count*100:.2f}%)")
print(f"Missing Station Info: {missing_station} ({missing_station/total_count*100:.2f}%)")
print("=" * 60)

# Weather category distribution
print("\nWeather Category Distribution:")
validation_df.groupBy("weather_category").count().orderBy("weather_category").show()

# Weather metrics statistics
print("\nWeather Metrics Statistics:")
validation_df.filter(
    (F.col("has_valid_temperature") == True) & 
    (F.col("has_valid_humidity") == True) &
    (F.col("has_valid_wind_speed") == True)
).select(
    F.min("temperature_celsius").alias("min_temp"),
    F.avg("temperature_celsius").alias("avg_temp"),
    F.max("temperature_celsius").alias("max_temp"),
    F.avg("humidity_percent").alias("avg_humidity"),
    F.avg("wind_speed_kmh").alias("avg_wind_speed")
).show()

# Show sample of flagged records
print("\nSample of records with data quality issues:")
validation_df.filter(
    (F.col("has_valid_temperature") == False) | 
    (F.col("has_valid_humidity") == False) |
    (F.col("has_valid_location") == False) |
    (F.col("has_valid_timestamp") == False)
).select(
    "station_id", "station_name", "temperature_celsius", "humidity_percent",
    "location_lat", "location_lon", "timestamp",
    "has_valid_temperature", "has_valid_humidity", "has_valid_location", "has_valid_timestamp"
).show(10, truncate=False)